# 🛵 Notebook 2: Sidecar as a separate process

In real deployments the sidecar is its **own process**, running on the same host (or the same Kubernetes pod). The app and the sidecar share the network namespace, so they talk to each other over **localhost** — fast, and invisible to the outside world.

We'll simulate this with two tiny HTTP servers on the same machine:

```
             client
               |  http :9000  (only port exposed)
               v
     +--------------+   localhost    +--------------+
     |   sidecar    | -------------> |     app      |
     |    :9000     |     :9001      |    :9001     |
     | auth/log/etc |                | business only|
     +--------------+                +--------------+
      (same pod / same host — shared network namespace)
```

The client never talks to the app directly. The sidecar is the **only door**.

## 🛠️ Setup

```bash
cd 05-microservices/sidecar
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

This notebook uses only the Python standard library. If you re-run a cell, the helpers below reuse the same thread so you won't get an "address already in use" error.

## Shared helpers
A tiny server that can be safely started once and reused across cell re-runs.

In [ ]:
import threading, time, json, urllib.request, urllib.error
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

# Track servers so we only start each one once per kernel session.
_servers = {}

def start_server(name, port, handler_cls):
    """Start an HTTP server in a background thread (idempotent)."""
    if name in _servers:
        print(f'{name} already running on :{_servers[name].server_port}')
        return _servers[name]
    srv = ThreadingHTTPServer(('127.0.0.1', port), handler_cls)
    threading.Thread(target=srv.serve_forever, daemon=True).start()
    _servers[name] = srv
    time.sleep(0.2)
    print(f'{name} up on :{port}')
    return srv

## The app — only business logic, binds to 127.0.0.1

Two important details:
1. **Binds to `127.0.0.1`**, not `0.0.0.0`. In a pod, only things sharing the pod's network can reach it — which means only the sidecar. External traffic *must* go through the sidecar.
2. **No auth, no log, no metric code.** Pure business logic.

In [ ]:
class App(BaseHTTPRequestHandler):
    def do_GET(self):
        body = json.dumps({'msg': 'hello from app', 'path': self.path}).encode()
        self.send_response(200)
        self.send_header('Content-Type', 'application/json')
        self.send_header('Content-Length', str(len(body)))
        self.end_headers()
        self.wfile.write(body)
    def log_message(self, *a, **kw): pass  # silence default access log

start_server('app', 9001, App)

## The sidecar — auth + logs + latency metric, forwards to the app

It owns the cross-cutting concerns. The app is free to stay small.

In a real service mesh (Envoy, linkerd-proxy), this is the same role — just with more features (mTLS, retries, circuit breaking, tracing) and configured by a control plane.

In [ ]:
class Sidecar(BaseHTTPRequestHandler):
    UPSTREAM = 'http://127.0.0.1:9001'
    ALLOWED  = {'secret'}   # policy, owned by the sidecar — not by the app

    def do_GET(self):
        # 1) auth
        if self.headers.get('X-Token') not in self.ALLOWED:
            self.send_response(401); self.end_headers(); self.wfile.write(b'unauthorized'); return
        # 2) call upstream, measuring latency
        t0 = time.time()
        try:
            with urllib.request.urlopen(f'{self.UPSTREAM}{self.path}', timeout=2) as r:
                status, body = r.status, r.read()
        except Exception as e:
            print(f'[sidecar] upstream error: {e}')
            self.send_response(502); self.end_headers(); self.wfile.write(b'bad gateway'); return
        dur = time.time() - t0
        # 3) log + forward response
        print(f'[sidecar] {self.command} {self.path} -> {status} ({dur*1000:.1f} ms)')
        self.send_response(status)
        self.send_header('Content-Type', 'application/json')
        self.send_header('Content-Length', str(len(body)))
        self.end_headers()
        self.wfile.write(body)
    def log_message(self, *a, **kw): pass

start_server('sidecar', 9000, Sidecar)

## Call it end-to-end
The client only knows about `:9000` (the sidecar).

In [ ]:
def call(token, path='/ping'):
    req = urllib.request.Request(f'http://127.0.0.1:9000{path}', headers={'X-Token': token})
    try:
        return urllib.request.urlopen(req, timeout=2).read().decode()
    except urllib.error.HTTPError as e:
        return f'HTTP {e.code} {e.read().decode()}'

print('good token:', call('secret', '/ping'))
print('bad  token:', call('wrong', '/ping'))
print('other path:', call('secret', '/hello'))

## Can clients bypass the sidecar?

In this toy the app binds to `127.0.0.1:9001`, so **only processes on the same machine** can reach it. In a Kubernetes pod the app bind would be `127.0.0.1`, and only pod-local processes (= the sidecar) can call it — external traffic must go through `:9000`.

Inside *this notebook* the sidecar and the "client" live in the same Python process, so a local call to 9001 will succeed — that's a property of the simulation, not a flaw of the pattern.

In [ ]:
# Same machine, so this works here -- in a real pod it would not.
print('direct to app (simulation):', urllib.request.urlopen('http://127.0.0.1:9001/ping', timeout=2).read().decode())

## Proving the headline claim: change policy, don't touch the app

"You can swap the sidecar without redeploying the app" is the sentence every sidecar
article ends on, and it's usually left as an assertion. Let's actually do it.

The app process below is **never restarted** and its code is **never edited**. We only
change the sidecar's policy — and the externally observable behaviour of the service
changes with it.

In [ ]:
# The app is still the same process started way back in the 'app' cell.
print('app server object id (unchanged throughout):', id(_servers['app']))

# --- policy v1: static shared secret (what the sidecar has been enforcing) ---
print('\n[policy v1: shared secret]')
print('  token=secret  ->', call('secret', '/ping')[:40])
print('  token=wrong   ->', call('wrong',  '/ping'))

# --- rotate the credential + add a response header, WITHOUT touching the app ---
Sidecar.ALLOWED = {'rotated-token-v2'}

def do_GET_v2(self):
    if self.headers.get('X-Token') not in Sidecar.ALLOWED:
        self.send_response(401); self.end_headers()
        self.wfile.write(b'unauthorized'); return
    with urllib.request.urlopen(f'{self.UPSTREAM}{self.path}', timeout=2) as r:
        body = r.read()
    self.send_response(200)
    self.send_header('Content-Type', 'application/json')
    self.send_header('X-Served-Via', 'sidecar-policy-v2')   # new behaviour
    self.send_header('Content-Length', str(len(body)))
    self.end_headers(); self.wfile.write(body)

Sidecar.do_GET = do_GET_v2      # stands in for "deploy a new sidecar image"

print('\n[policy v2: rotated credential + new header] — app untouched')
print('  old token=secret          ->', call('secret', '/ping'))
print('  new token=rotated-token-v2 ->', call('rotated-token-v2', '/ping')[:40])

req = urllib.request.Request('http://127.0.0.1:9000/ping',
                             headers={'X-Token': 'rotated-token-v2'})
with urllib.request.urlopen(req, timeout=2) as r:
    print('  response header X-Served-Via =', r.headers.get('X-Served-Via'))
print('\napp server object id (still unchanged):', id(_servers['app']))

Credential rotation and a new response header shipped with **zero** application
changes and **zero** application downtime. In Kubernetes this is literally a new
sidecar image tag in the pod spec.

> Two honest caveats. Replacing a method on a live class is a notebook trick — in
> production the sidecar is a separate container and the swap is a rolling restart of
> *that* container. And the swap is only free while the app→sidecar contract (localhost
> port, protocol) stays the same; change that and you are back to a coordinated deploy.

### Takeaways
- The **app never checked auth or wrote a log line** — the sidecar did.
- They talk over **localhost**, so the extra hop is cheap (sub-millisecond in real infra).
- You can **swap the sidecar** (new TLS cert, new log format, new retry policy) without redeploying the app container.
- This is exactly how **Envoy + Istio / linkerd-proxy** operate in a service mesh — just with much richer features and a control plane to configure them at fleet scale.

➡️ Notebook 3 shows real-world sidecar patterns: retries with timeouts, a log-forwarder sidecar, metrics, and trade-offs.